In [1]:
import importlib
from pathlib import Path
import pandas as pd
import src.collect_licences as cl

importlib.reload(cl)

chemin_parquet = Path("data/processed/licences_dep.parquet")

if chemin_parquet.exists():
    df_licences = pd.read_parquet(chemin_parquet)
else:
    df_licences = cl.construire_base_departementale()

print("Années disponibles :", sorted(df_licences["annee"].unique()))
print("Exemples de fédérations :", sorted(df_licences["nom_fed"].dropna().unique())[:15])

Années disponibles : [np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023)]
Exemples de fédérations : ['F Nationale du Sport en Milieu Rural', 'F Nautique de Pêche Sportive  en Apnée', "F Sportive Educative de l'Enseignement Catholique (UGSEL)", 'F Sportive de la Police Nationale', 'F Sportive des ASPTT', 'F Sportive et Culturelle de France', 'F Sportive et Gymnique du Travail', 'F de Boxe Américaine et DA', 'F de Double Dutch', 'F de Voitures Radio Commandées', 'F des Arts Énergétiques et Martiaux Chinois', 'F des Clubs de la Défense', 'F des Sports et Loisirs Canins', 'F nationale du sport en milieu rural', 'F sportive et culturelle de France']


In [6]:
from src.processing import (
    preparer_serie_base_100,
    calculer_ecart_double_difference,
    calculer_croissance_departementale,
)
from src.viz import plot_effet_local_base100, plot_carte_croissance_dep

# Exemple : Tennis de table et les frères Lebrun (Hérault - 34)
CODE_FFTT = "124"

df_ping_100 = preparer_serie_base_100(
    df_licences,
    federation=CODE_FFTT,
    dep_cible="34",
    annee_ref=2019,
    label_cible="Hérault (34 - Fief Lebrun)",
    col_fed="code_fed",
)

ecart_did = calculer_ecart_double_difference(
    df_ping_100,
    label_cible="Hérault (34 - Fief Lebrun)",
    annee_avant=2016,
    annee_apres=2023,
)
print(f"Surplus de croissance dans l'Hérault (2019-2023) : {ecart_did:+.1f} points d'indice")

fig_courbe = plot_effet_local_base100(
    df_ping_100, "Tennis de table", "Frères Lebrun", annee_ref=2019, annee_evenement=2021
)
fig_courbe.show()

df_ping_carte = calculer_croissance_departementale(
    df_licences,
    federation=CODE_FFTT,
    annee_debut=2016,
    annee_fin=2023,
    col_fed="code_fed",
)
fig_carte = plot_carte_croissance_dep(df_ping_carte, "Tennis de table", annee_debut=2016, annee_fin=2023)
fig_carte.show()

Surplus de croissance dans l'Hérault (2019-2023) : +2.2 points d'indice


In [7]:
import importlib
from pathlib import Path
import pandas as pd
import src.collect_licences as cl
from src.processing import (
    preparer_serie_base_100,
    calculer_ecart_double_difference,
    calculer_croissance_departementale,
)
from src.viz import plot_effet_local_base100, plot_carte_croissance_dep

importlib.reload(cl)

# 1. Chargement ou génération de la base départementale 2016-2025
chemin_complet = Path("data/processed/licences_dep_2016_2025.parquet")
if chemin_complet.exists():
    df_licences = pd.read_parquet(chemin_complet)
else:
    df_hist = pd.read_parquet("data/processed/licences_dep.parquet")
    df_licences = cl.enrichir_avec_2025(df_hist)

print("Années disponibles :", sorted(df_licences["annee"].unique()))

# 2. Vérifie les codes fédérations (ex. FFTT, Natation, Basket, Judo, Cyclisme)
mots_cles = ["tennis de table", "natation", "basket", "judo", "cyclisme"]
masque = df_licences["nom_fed"].str.lower().str.contains("|".join(mots_cles), na=False)
display(df_licences[masque][["code_fed", "nom_fed"]].drop_duplicates().sort_values("code_fed"))

Base 2016-2025 sauvegardée dans data/processed/licences_dep_2016_2025.parquet (86262 lignes).
Années disponibles : [np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2025)]


,code_fed,nom_fed
3,105,Fédération française de basketball
9453,105,Fédération Française de basketball
18188,105,FF de basketball
26853,105,FF de Basketball
73505,105,Fédération Française de Basketball
6,108,Fédération française de cyclisme
9456,108,Fédération Française de cyclisme
18190,108,FF de cyclisme
26856,108,FF de Cyclisme
73508,108,Fédération Française de Cyclisme


In [14]:
CODE_FFTT = "124"  # À ajuster selon le tableau ci-dessus

df_ping_100 = preparer_serie_base_100(
    df_licences,
    federation=CODE_FFTT,
    dep_cible="34",
    annee_ref=2019,
    label_cible="Hérault (34 - Fief Lebrun)",
    col_fed="code_fed",
)

ecart_did = calculer_ecart_double_difference(
    df_ping_100, label_cible="Hérault (34 - Fief Lebrun)", annee_avant=2019, annee_apres=2025
)
print(f"Surplus de croissance dans l'Hérault (2023-2025) : {ecart_did:+.1f} points d'indice")

fig_courbe = plot_effet_local_base100(df_ping_100, "Tennis de table", "Frères Lebrun", annee_ref=2019, annee_evenement=2024.5)
fig_courbe.show()

df_ping_carte = calculer_croissance_departementale(
    df_licences, federation=CODE_FFTT, annee_debut=2019, annee_fin=2025, col_fed="code_fed"
)
fig_carte = plot_carte_croissance_dep(df_ping_carte, "Tennis de table", annee_debut=2019, annee_fin=2025)
fig_carte.show()

Surplus de croissance dans l'Hérault (2023-2025) : +15.1 points d'indice


In [ ]:
CODE_FFTT = "124"  # À ajuster selon le tableau ci-dessus

df_ping_100 = preparer_serie_base_100(
    df_licences,
    federation=CODE_FFTT,
    dep_cible="34",
    annee_ref=2023,
    label_cible="Hérault (34 - Fief Lebrun)",
    col_fed="code_fed",
)

ecart_did = calculer_ecart_double_difference(
    df_ping_100, label_cible="Hérault (34 - Fief Lebrun)", annee_avant=2023, annee_apres=2025
)
print(f"Surplus de croissance dans l'Hérault (2023-2025) : {ecart_did:+.1f} points d'indice")

fig_courbe = plot_effet_local_base100(df_ping_100, "Tennis de table", "Frères Lebrun", annee_ref=2023, annee_evenement=2024.5)
# 2024 correspond a septembre 2023- juin 2024 donc pas encore l'effet des JO (que de l'anticipation, mais osef ici)
fig_courbe.show()

df_ping_carte = calculer_croissance_departementale(
    df_licences, federation=CODE_FFTT, annee_debut=2023, annee_fin=2025, col_fed="code_fed"
)
fig_carte = plot_carte_croissance_dep(df_ping_carte, "Tennis de table", annee_debut=2023, annee_fin=2025)
fig_carte.show()

Surplus de croissance dans l'Hérault (2023-2025) : -0.3 points d'indice


### Analyse de l'effet « Héros local » : choix de l'année de référence (2019 vs 2023)

Le choix de l'année de référence conditionne l'hypothèse testée et permet de décomposer la diffusion de l'effet « frères Lebrun » en deux temps :

* **Base 100 en 2019 (Référence principale : effet global du rôle modèle)**
  * **Justification :** 2019 correspond à la dernière saison normale avant la crise sanitaire et avant l'émergence médiatique des frères Lebrun. En l'absence de choc local, l'Hérault (34) aurait dû suivre la même trajectoire de reprise que le reste de la France.
  * **Résultats :** On observe un surplus de croissance net de **+15 points d'indice** dans l'Hérault en 2025 (indice 120 contre 105 au niveau national). Cet écart se creuse dès la période pré-olympique 2021-2023 (indice 96 contre 84), portée par leurs premiers titres nationaux et européens.

* **Base 100 en 2023 (Vérification : effet marginal des JO de Paris 2024)**
  * **Justification :** Permet d'isoler l'impact pur des Jeux Olympiques de Paris 2024 une fois les athlètes déjà connus localement.
  * **Résultats :** Entre 2023 et 2025, les courbes de l'Hérault et du reste de la France se superposent (+25 % sur l'ensemble du territoire). L'exposition olympique a transformé un engouement initialement régional en un phénomène national homogène.

#### Limites méthodologiques à retenir
1. **Tendances pré-événement (2016-2019) :** L'Hérault affichait une légère érosion avant 2019 (passant de l'indice 107 à 100) contre une stabilité nationale, ce qui renforce la portée du retournement observé après 2021.
2. **Amplitude du creux Covid en 2021 :** La chute plus marquée des licences héraultaises en 2021 (indice 50 contre 60 au niveau national) explique une partie de la vigueur du rebond initial, bien que le dépassement durable du seuil de 2019 confirme un véritable effet d'entraînement.

### GENRE

In [ ]:
import importlib
import src.processing as pr
import src.viz as vz

importlib.reload(pr)
importlib.reload(vz)

df_ping_genre = pr.analyser_effet_genre(df_licences, federation="124", annee_ref=2019, col_fed="code_fed")
display(df_ping_genre[["annee", "licences_f", "licences_h", "taux_feminisation", "indice_f", "indice_h"]])

fig_genre = vz.plot_effet_genre(df_ping_genre, "Tennis de table", "Frères Lebrun", annee_ref=2019)
fig_genre.show()

,annee,licences_f,licences_h,taux_feminisation,indice_f,indice_h
0,2016,38767,166897.0,18.849677,92.502804,98.425982
1,2017,38358,165983.0,18.771563,91.526880,97.886958
2,2018,40199,166444.0,19.453357,95.919731,98.158829
3,2019,41909,169566.0,19.817473,100.000000,100.000000
4,2020,21783,136215.0,13.786883,51.976902,80.331552
5,2021,16436,109742.0,13.026043,39.218306,64.719342
6,2022,23722,134391.0,15.003194,56.603594,79.255865
7,2023,26953,150460.0,15.192235,64.313155,88.732411
8,2025,32364,189898.0,14.561194,77.224463,111.990611


### TELLEMENT D HOMME 
+20K DE LICENCIES HOMME PAR RAPPORT A 2019
-7K PR LES FEMMES :/

### Effet JO
* amplifie l'ecart
Nuance méthodologique (limite à noter) : On observe une chute brutale des licences féminines dès 2020 (-48 % contre -20 % chez les hommes), liée à la fois à l'arrêt de la pratique loisir pendant la crise sanitaire et à une évolution des licences promotionnelles/scolaires dans les statistiques INJEP à partir de 2019-2020. L'engouement suscité par les frères Lebrun (deux modèles masculins) a néanmoins accéléré le rattrapage masculin bien plus fortement que le rattrapage féminin.

==> Il faudra comp avec la natation (LM) + judo/cyclisme avec les femmes

## AGE

In [24]:
import importlib
import src.collect_licences as cl
import src.viz as vz

importlib.reload(cl)
importlib.reload(vz)

df_ping_ages = cl.construire_base_ages_2019_2025(code_fed_cible="124")

# Part de chaque tranche d'âge dans les nouveaux licenciés gagnés entre 2023 et 2025
piv_age = df_ping_ages.pivot(index="tranche_age", columns="annee", values="licences")
piv_age["gain_2023_2025"] = piv_age[2025] - piv_age[2023]
piv_age["part_du_gain (%)"] = (piv_age["gain_2023_2025"] / piv_age["gain_2023_2025"].sum()) * 100
display(piv_age)

fig_age = vz.plot_effet_age(df_ping_ages, "Tennis de table", "Frères Lebrun", annee_ref=2023)
fig_age.show()

annee,2019,2020,2021,2022,2023,2025,gain_2023_2025,part_du_gain (%)
tranche_age,,,,,,,,
10 à 14 ans,55959.0,42359.0,32160.0,41470.0,47406.0,60259.0,12853.0,28.658387
15 à 19 ans,15905.0,11883.0,10987.0,12345.0,12838.0,14250.0,1412.0,3.148342
20 ans et plus,97561.0,83391.0,70603.0,84583.0,95196.0,121262.0,26066.0,58.119468
Moins de 10 ans,41928.0,20272.0,12318.0,19715.0,21973.0,26491.0,4518.0,10.073803


### Analyse de l'effet Âge (Base 100 en 2023) : un engouement généralisé

Le recalibrage sur **2023** permet d'isoler l'impact des Jeux Olympiques de Paris 2024 et des frères Lebrun sans le biais du décrochage des licences scolaires en 2020. Entre 2023 et 2025, la fédération gagne **+44 849 licenciés** et toutes les classes d'âge affichent une croissance à deux chiffres :

* **Deux moteurs à quasi-égalité en rythme de croissance (+27 %) :**
  * Les **20 ans et plus** progressent de **+27,4 %** (indice 127,4) et représentent **58,1 %** du gain total (+26 066 licenciés), portés par les jeunes actifs et les pratiquants loisirs ou anciens joueurs revenant en club.
  * Les **10 à 14 ans** suivent exactement la même dynamique avec **+27,1 %** (indice 127,1), générant **28,7 %** des nouveaux inscrits (+12 853 licenciés). Cette tranche d'âge incarne le cœur de l'identification aux frères Lebrun.
* **Une reprise confirmée chez les plus jeunes (+20,6 %) :** Les **moins de 10 ans** atteignent l'indice **120,6** (+4 518 licenciés, soit 10,1 % de la hausse). Bien qu'ils restent en deçà de leur volume d'avant-Covid en raison de la baisse structurelle des licences scolaires depuis 2020, leur rebond post-JO est bien réel.
* **Un effet plus modéré chez les grands adolescents (+11,0 %) :** Les **15 à 19 ans** progressent plus lentement (indice 111,0, soit 3,1 % du gain total), une inertie habituelle liée aux contraintes des examens et des études supérieures.

**Conclusion intermédiaire :** L'effet « Paris 2024 et frères Lebrun » ne se limite pas au public enfantin : les **moins de 20 ans** représentent **41,9 %** de la croissance totale, mais la vague d'adhésions touche avec la même intensité relative les collégiens (10-14 ans) et les adultes.

In [25]:
# pour avoir les ref des sports

mots_cles = ["tennis de table", "natation", "basket", "judo", "cyclisme"]
masque = df_licences["nom_fed"].str.lower().str.contains("|".join(mots_cles), na=False)

correspondance = (
    df_licences[masque][["code_fed", "nom_fed", "annee"]]
    .groupby(["code_fed", "nom_fed"])["annee"]
    .agg(["min", "max", "count"])
    .reset_index()
)
display(correspondance)

,code_fed,nom_fed,min,max,count
0,105,FF de Basketball,2019,2023,496
1,105,FF de basketball,2018,2018,101
2,105,Fédération Française de Basketball,2025,2025,110
3,105,Fédération Française de basketball,2017,2017,101
4,105,Fédération française de basketball,2016,2016,101
5,108,FF de Cyclisme,2019,2023,496
6,108,FF de cyclisme,2018,2018,101
7,108,Fédération Française de Cyclisme,2025,2025,110
8,108,Fédération Française de cyclisme,2017,2017,101
9,108,Fédération française de cyclisme,2016,2016,101
